# CrossLLM — Qwen3.8-27B campaign runner

This notebook reads the model and campaign ZIP directly from Google Drive, copies the model to Colab's local SSD, then runs resumable proposal generation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/khoilv2005/CrosLLM.git /content/CrosLLM
%cd /content/CrosLLM
!pip -q install -r colab/qwen38_27b/requirements-colab.txt

In [ ]:
from pathlib import Path

MODEL_DRIVE_PATH = Path('/content/drive/MyDrive/CrossLLM/models/Qwen3.8-27B')
DATASET_ZIP_PATH = Path('/content/drive/MyDrive/CrossLLM/dataset/qwen38_27b_campaigns.zip')
DRIVE_ROOT = Path('/content/drive/MyDrive/CrossLLM')
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints' / 'qwen38_27b'
MAX_CAMPAIGNS = 2  # increase after the smoke test

assert (MODEL_DRIVE_PATH / 'config.json').exists(), f'Model not found: {MODEL_DRIVE_PATH}'
assert DATASET_ZIP_PATH.exists(), f'Dataset ZIP not found: {DATASET_ZIP_PATH}'
print('Model:', MODEL_DRIVE_PATH)
print('Dataset ZIP:', DATASET_ZIP_PATH)
print('Checkpoints:', CHECKPOINT_DIR)

In [ ]:
import os, shutil, zipfile

dataset_dir = DATASET_ZIP_PATH.parent / 'qwen38_27b_extracted'
campaigns_path = dataset_dir / 'campaigns.jsonl'
if not campaigns_path.exists():
    dataset_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP_PATH) as archive:
        archive.extractall(dataset_dir)
assert campaigns_path.exists(), f'campaigns.jsonl missing in {DATASET_ZIP_PATH}'

local_model_path = Path('/content/models/Qwen3.8-27B')
if not local_model_path.exists():
    shutil.copytree(MODEL_DRIVE_PATH, local_model_path, ignore=shutil.ignore_patterns('.git'))
assert (local_model_path / 'config.json').exists()
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
os.environ['CAMPAIGNS_PATH'] = str(campaigns_path)
os.environ['LOCAL_MODEL_PATH'] = str(local_model_path)
os.environ['CHECKPOINT_DIR'] = str(CHECKPOINT_DIR)
print('Campaigns:', campaigns_path)
print('Local model:', local_model_path)
print('Checkpoint directory:', CHECKPOINT_DIR)

In [ ]:
!python colab/qwen38_27b/run_local_qwen38_campaigns.py --campaigns "$CAMPAIGNS_PATH" --model-path "$LOCAL_MODEL_PATH" --out-dir "$CHECKPOINT_DIR" --max-campaigns $MAX_CAMPAIGNS --slots-per-campaign 8
# Re-run this cell to resume. Completed campaigns and successful slots are skipped automatically.